# 2011 Local Government Election Data Cleaning

This notebook prepares the 2011 local government election results for analysis. It loads the source CSV, standardizes column names, selects the required fields, converts text values into numeric types, removes excluded ballot records, aggregates results, calculates municipality-level turnout, and exports the cleaned data.

## Import pandas

`pandas` supplies the DataFrame operations used to inspect, transform, group, and export the election data.

In [1]:
# Import pandas for tabular data cleaning and aggregation.
import pandas as pd
from pathlib import Path

# Paths are relative to the project folder, so this notebook runs on any computer.
# It works whether Jupyter is opened in the notebooks/ folder or in the project root.
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_DIR / "data" / "raw"
CLEAN_DIR = PROJECT_DIR / "data" / "cleaned"

## Load the raw 2011 data

Read the source CSV from `data/raw/` using `ISO-8859-1` encoding so that accented or special characters in the election data are handled correctly. Display the DataFrame to inspect the imported records.

In [2]:
# Load the raw election results with an encoding that supports special characters.
df = pd.read_csv(RAW_DIR / "2011_LGE.csv.zip", encoding="ISO-8859-1")

# Display the imported data for an initial review.
df

/tmp/ipykernel_172/1294307036.py:2: DtypeWarning: Columns (0: MEC7
Votes, 1: Spoilt
Votes) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(RAW_DIR / "2011_LGE.csv.zip", encoding="ISO-8859-1")


,Electoral Event,Province,Municipality,Ward,Voting \nDistrict,Party,Ballot \nType,Registered\nVoters,% Voter \nTurnout,MEC7\nVotes,Total Votes \nCast,Valid Votes \nCast,Spoilt\nVotes
0,Local Government Elections 2011,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,29200001,10590151,AFRICAN CHRISTIAN DEMOCRATIC PARTY,PR,"2,421",49.94%,4,"1,205",5,27
1,Local Government Elections 2011,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,29200001,10590151,AFRICAN CHRISTIAN DEMOCRATIC PARTY,WARD,"2,421",49.94%,0,"1,190",5,23
2,Local Government Elections 2011,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,29200001,10590151,AFRICAN INDEPENDENT CONGRESS,PR,"2,421",49.94%,4,"1,205",42,27
3,Local Government Elections 2011,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,29200001,10590151,AFRICAN NATIONAL CONGRESS,PR,"2,421",49.94%,4,"1,205",415,27
4,Local Government Elections 2011,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,29200001,10590151,AFRICAN NATIONAL CONGRESS,WARD,"2,421",49.94%,0,"1,190",444,23
...,...,...,...,...,...,...,...,...,...,...,...,...,...
474381,Local Government Elections 2011,Western Cape,WC053 - Beaufort West [Beaufort West],10503007,98180073,NATIONAL PEOPLE'S PARTY,PR,107,28.97%,0,31,0,0
474382,Local Government Elections 2011,Western Cape,WC053 - Beaufort West [Beaufort West],10503007,98180073,NATIONAL PEOPLE'S PARTY,WARD,107,28.97%,0,31,0,0
474383,Local Government Elections 2011,Western Cape,WC053 - Beaufort West [Beaufort West],10503007,98180073,SOUTH AFRICAN PROGRESSIVE CIVIC ORGANISATION,DC 40%,107,28.97%,0,31,0,0
474384,Local Government Elections 2011,Western Cape,WC053 - Beaufort West [Beaufort West],10503007,98180073,SOUTH AFRICAN PROGRESSIVE CIVIC ORGANISATION,PR,107,28.97%,0,31,0,0


## Inspect the source columns

Review the original column names before cleaning them. This identifies formatting issues and confirms which fields are available.

In [3]:
# Inspect the original column names before standardizing them.
df.columns

Index(['Electoral Event', 'Province', 'Municipality', 'Ward',
       'Voting \nDistrict', 'Party', 'Ballot \nType', 'Registered\nVoters',
       '% Voter \nTurnout', 'MEC7\nVotes', 'Total Votes \nCast',
       'Valid Votes \nCast', 'Spoilt\nVotes'],
      dtype='str')

## Clean column names

Remove embedded newline characters from every column name so the fields can be referenced consistently in later operations.

In [4]:
# Remove newline characters embedded in the source column labels.
df = df.rename(columns=lambda c: c.replace('\n', ''))

# Confirm the cleaned column names.
df.columns

Index(['Electoral Event', 'Province', 'Municipality', 'Ward',
       'Voting District', 'Party', 'Ballot Type', 'RegisteredVoters',
       '% Voter Turnout', 'MEC7Votes', 'Total Votes Cast', 'Valid Votes Cast',
       'SpoiltVotes'],
      dtype='str')

## Select the analysis fields

Create a smaller working DataFrame containing the geographic identifiers, party, ballot type, turnout, and valid vote count needed for the analysis.

In [5]:
# Keep only the fields needed for the cleaning and aggregation steps.
draft = df.filter(axis=1, items=['Province','Municipality','Party','Ballot Type','% Voter Turnout','Valid Votes Cast'])

## Inspect the working DataFrame

Check the selected columns, data types, and non-null counts before converting the text-based numeric fields.

In [6]:
# Check the selected fields and their current data types.
draft.info()

<class 'pandas.DataFrame'>
RangeIndex: 474386 entries, 0 to 474385
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   Province          474386 non-null  str  
 1   Municipality      474386 non-null  str  
 2   Party             474386 non-null  str  
 3   Ballot Type       474386 non-null  str  
 4   % Voter Turnout   474386 non-null  str  
 5   Valid Votes Cast  474386 non-null  str  
dtypes: str(6)
memory usage: 21.7 MB


## Convert turnout and vote totals to numbers

Remove the percent sign from turnout values and convert them to floating-point numbers. Remove thousands separators from valid vote totals and convert them to integers.


Display the converted DataFrame to confirm that the values are ready for aggregation.

In [7]:
# Remove the percent sign and convert turnout to a numeric percentage.
draft['% Voter Turnout'] = draft['% Voter Turnout'].str.removesuffix('%').astype('float')

# Remove thousands separators and convert vote totals to integers.
draft['Valid Votes Cast'] = draft['Valid Votes Cast'].str.replace(',',"", regex=False).astype('int')

# Display the converted values before filtering and aggregation.
draft

,Province,Municipality,Party,Ballot Type,% Voter Turnout,Valid Votes Cast
0,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN CHRISTIAN DEMOCRATIC PARTY,PR,49.94,5
1,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN CHRISTIAN DEMOCRATIC PARTY,WARD,49.94,5
2,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN INDEPENDENT CONGRESS,PR,49.94,42
3,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN NATIONAL CONGRESS,PR,49.94,415
4,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN NATIONAL CONGRESS,WARD,49.94,444
...,...,...,...,...,...,...
474381,Western Cape,WC053 - Beaufort West [Beaufort West],NATIONAL PEOPLE'S PARTY,PR,28.97,0
474382,Western Cape,WC053 - Beaufort West [Beaufort West],NATIONAL PEOPLE'S PARTY,WARD,28.97,0
474383,Western Cape,WC053 - Beaufort West [Beaufort West],SOUTH AFRICAN PROGRESSIVE CIVIC ORGANISATION,DC 40%,28.97,0
474384,Western Cape,WC053 - Beaufort West [Beaufort West],SOUTH AFRICAN PROGRESSIVE CIVIC ORGANISATION,PR,28.97,0


## Check for duplicate rows

Inspect duplicate records without assigning the de-duplicated result back to `draft`. The second display shows the current working DataFrame.

In [8]:
# Inspect duplicates without assigning the result back to draft.
draft.drop_duplicates()

# Display the current working DataFrame.
draft

,Province,Municipality,Party,Ballot Type,% Voter Turnout,Valid Votes Cast
0,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN CHRISTIAN DEMOCRATIC PARTY,PR,49.94,5
1,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN CHRISTIAN DEMOCRATIC PARTY,WARD,49.94,5
2,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN INDEPENDENT CONGRESS,PR,49.94,42
3,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN NATIONAL CONGRESS,PR,49.94,415
4,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN NATIONAL CONGRESS,WARD,49.94,444
...,...,...,...,...,...,...
474381,Western Cape,WC053 - Beaufort West [Beaufort West],NATIONAL PEOPLE'S PARTY,PR,28.97,0
474382,Western Cape,WC053 - Beaufort West [Beaufort West],NATIONAL PEOPLE'S PARTY,WARD,28.97,0
474383,Western Cape,WC053 - Beaufort West [Beaufort West],SOUTH AFRICAN PROGRESSIVE CIVIC ORGANISATION,DC 40%,28.97,0
474384,Western Cape,WC053 - Beaufort West [Beaufort West],SOUTH AFRICAN PROGRESSIVE CIVIC ORGANISATION,PR,28.97,0


## Exclude DC 40% records

Remove `DC 40%` ballot records before calculating grouped totals and mean turnout, keeping the analysis focused on the intended ballot categories.

In [9]:
# Exclude DC 40% ballot records from the analysis.
draft = draft[draft['Ballot Type']!="DC 40%"]

# Review the filtered records.
draft

,Province,Municipality,Party,Ballot Type,% Voter Turnout,Valid Votes Cast
0,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN CHRISTIAN DEMOCRATIC PARTY,PR,49.94,5
1,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN CHRISTIAN DEMOCRATIC PARTY,WARD,49.94,5
2,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN INDEPENDENT CONGRESS,PR,49.94,42
3,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN NATIONAL CONGRESS,PR,49.94,415
4,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN NATIONAL CONGRESS,WARD,49.94,444
...,...,...,...,...,...,...
474379,Western Cape,WC053 - Beaufort West [Beaufort West],INDEPENDENT CONGRESS,WARD,28.97,0
474381,Western Cape,WC053 - Beaufort West [Beaufort West],NATIONAL PEOPLE'S PARTY,PR,28.97,0
474382,Western Cape,WC053 - Beaufort West [Beaufort West],NATIONAL PEOPLE'S PARTY,WARD,28.97,0
474384,Western Cape,WC053 - Beaufort West [Beaufort West],SOUTH AFRICAN PROGRESSIVE CIVIC ORGANISATION,PR,28.97,0


## Aggregate records by party and ballot type

Group the filtered records by province, municipality, party, and ballot type. Sum valid votes and calculate the mean turnout across the ward-level records in each group.

In [10]:
# Aggregate vote totals and turnout by geography, party, and ballot type.
draft = (
    draft
    .groupby(['Province', 'Municipality','Party','Ballot Type'], as_index=False)
    .agg(
        ValidVotesCast=('Valid Votes Cast', 'sum'),
        MeanVoterTurnout=('% Voter Turnout', 'mean') # Turnout from each ward
    )
)

# This display shows the grouped records before adding municipality-level turnout.
# draft = draft[draft['Ballot Type']!="DC 40%"]
draft

,Province,Municipality,Party,Ballot Type,ValidVotesCast,MeanVoterTurnout
0,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN CHRISTIAN DEMOCRATIC PARTY,PR,1485,58.126558
1,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN CHRISTIAN DEMOCRATIC PARTY,WARD,1687,57.993670
2,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN INDEPENDENT CONGRESS,PR,7616,58.126558
3,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN INDEPENDENT CONGRESS,WARD,355,58.975200
4,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN NATIONAL CONGRESS,PR,146919,58.126558
...,...,...,...,...,...,...
3276,Western Cape,WC053 - Beaufort West [Beaufort West],INDEPENDENT CONGRESS,WARD,84,60.998696
3277,Western Cape,WC053 - Beaufort West [Beaufort West],NATIONAL PEOPLE'S PARTY,PR,170,60.998696
3278,Western Cape,WC053 - Beaufort West [Beaufort West],NATIONAL PEOPLE'S PARTY,WARD,208,60.998696
3279,Western Cape,WC053 - Beaufort West [Beaufort West],SOUTH AFRICAN PROGRESSIVE CIVIC ORGANISATION,PR,40,60.998696


## Calculate municipality-level turnout

Average the grouped turnout values within each municipality to create one municipality-level turnout measure. Round the result to two decimal places for reporting.

In [11]:
# Average grouped turnout values to obtain one value per municipality.
municipality_turnout = (
    draft
    .groupby(['Municipality'], as_index=False)['MeanVoterTurnout']
    .mean()) # Turnout from each municipality

# Round municipality turnout percentages for consistent reporting.
municipality_turnout['MeanVoterTurnout'] = municipality_turnout['MeanVoterTurnout'].round(2)

# Review the municipality-level lookup table.
municipality_turnout

,Municipality,MeanVoterTurnout
0,BUF - Buffalo City Metropolitan Municipality [...,58.35
1,CPT - City of Cape Town [Cape Town],65.72
2,EC101 - Camdeboo [Graaff-Reinet],57.78
3,EC102 - Blue Crane Route [Somerset East],54.73
4,EC103 - Ikwezi [Jansenville],59.55
...,...,...
229,WC047 - Bitou [Greater Plettenberg Bay],74.08
230,WC048 - Knysna [Knysna],68.26
231,WC051 - Laingsburg [Laingsburg],76.68
232,WC052 - Prince Albert [Prins Albert],76.27


## Attach municipality turnout to each record

Remove the intermediate grouped turnout value and merge the municipality-level turnout lookup back into the aggregated records. This gives every party and ballot-type row its municipality's mean turnout.

In [12]:
# Replace grouped turnout with the municipality-level mean turnout.
draft = draft.drop(columns=['MeanVoterTurnout']).merge(
    municipality_turnout,
    on='Municipality',
    how='left'
 )

# Display the final cleaned records before export.
draft

,Province,Municipality,Party,Ballot Type,ValidVotesCast,MeanVoterTurnout
0,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN CHRISTIAN DEMOCRATIC PARTY,PR,1485,58.35
1,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN CHRISTIAN DEMOCRATIC PARTY,WARD,1687,58.35
2,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN INDEPENDENT CONGRESS,PR,7616,58.35
3,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN INDEPENDENT CONGRESS,WARD,355,58.35
4,Eastern Cape,BUF - Buffalo City Metropolitan Municipality [...,AFRICAN NATIONAL CONGRESS,PR,146919,58.35
...,...,...,...,...,...,...
3276,Western Cape,WC053 - Beaufort West [Beaufort West],INDEPENDENT CONGRESS,WARD,84,61.39
3277,Western Cape,WC053 - Beaufort West [Beaufort West],NATIONAL PEOPLE'S PARTY,PR,170,61.39
3278,Western Cape,WC053 - Beaufort West [Beaufort West],NATIONAL PEOPLE'S PARTY,WARD,208,61.39
3279,Western Cape,WC053 - Beaufort West [Beaufort West],SOUTH AFRICAN PROGRESSIVE CIVIC ORGANISATION,PR,40,61.39


## Export the cleaned dataset

Save the completed 2011 election dataset as `data/cleaned/2011_LGE_Cleaned.csv` for downstream analysis.

In [13]:
# Export the cleaned 2011 election data for downstream analysis.
draft.to_csv(CLEAN_DIR / '2011_LGE_Cleaned.csv')